# Benchmark – Results table & report views

Δημιουργεί το schema `analytics_dq_dev.benchmark`, τον πίνακα `comparison_results` και τα report views. Είναι idempotent: μπορεί να ξανατρέξει χωρίς να χαθούν δεδομένα.

* **report_inside_outside**: Inside the rules / Outside the rules / Total time ανά run (DQ vs GX).
* **report_rule_by_rule**: χρόνος ανά κανόνα ανά run. Όταν `dq_rules_batched = true`, το DQ έτρεξε όλους τους κανόνες σε ένα scan, άρα το `dq_sec` είναι κοινός χρόνος.

Το τελευταίο cell (TRUNCATE) σβήνει όλο το ιστορικό – τρέξ' το μόνο για καθαρή αρχή.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS analytics_dq_dev.benchmark;

CREATE TABLE IF NOT EXISTS analytics_dq_dev.benchmark.comparison_results (
    run_id             STRING,     -- Μοναδικό ID κάθε benchmark run
    run_timestamp      TIMESTAMP,  -- Πότε έγινε
    framework          STRING,     -- 'dq_framework' ή 'great_expectations'
    dataset            STRING,     -- Ποιο dataset χρησιμοποιήθηκε
    row_count          BIGINT,     -- Πόσες γραμμές είχε το dataset
    rule_name          STRING,     -- Όνομα κανόνα
    column_tested      STRING,     -- Ποια στήλη ελέγχθηκε
    rows_evaluated     BIGINT,
    rows_passed        BIGINT,
    rows_failed        BIGINT,
    rule_status        STRING,
    execution_time_sec DOUBLE,     -- Χρόνος εκτέλεσης κανόνα
    cluster_id         STRING,
    total_time_sec     DOUBLE      -- Συνολικός χρόνος framework (από την αρχή έως το τέλος)
)
USING DELTA
COMMENT 'Benchmark comparison results: DQ Framework vs Great Expectations';

In [0]:
%sql
CREATE OR REPLACE VIEW analytics_dq_dev.benchmark.report_inside_outside
COMMENT 'Benchmark report: time inside vs outside the rules, DQ Framework vs Great Expectations, per run'
AS
WITH per_fw AS (
  SELECT
    run_id,
    framework,
    MIN(run_timestamp) AS run_ts,
    -- Batched execution (one scan for all rules) gives every rule the same duration -> count it once
    CASE WHEN COUNT(*) > 1 AND COUNT(DISTINCT execution_time_sec) = 1
         THEN MAX(execution_time_sec) ELSE SUM(execution_time_sec) END AS inside_sec,
    MAX(total_time_sec) AS total_sec,
    MAX(row_count)      AS row_count,
    MAX(dataset)        AS dataset
  FROM analytics_dq_dev.benchmark.comparison_results
  GROUP BY run_id, framework
),
pivoted AS (
  SELECT
    run_id,
    MIN(run_ts) AS run_ts,
    MAX(CASE WHEN framework = 'great_expectations' THEN inside_sec END) AS gx_inside,
    MAX(CASE WHEN framework = 'dq_framework'       THEN inside_sec END) AS dq_inside,
    MAX(CASE WHEN framework = 'great_expectations' THEN total_sec END)  AS gx_total,
    MAX(CASE WHEN framework = 'dq_framework'       THEN total_sec END)  AS dq_total,
    MAX(row_count) AS row_count,
    MAX(dataset)   AS dataset
  FROM per_fw
  GROUP BY run_id
),
labeled AS (
  SELECT *, ROW_NUMBER() OVER (ORDER BY run_ts) AS exec_no FROM pivoted
)
SELECT
  exec_no,
  CONCAT('Exec ', exec_no)                       AS exec_label,
  run_id,
  run_ts,
  dataset,
  row_count,
  CONCAT(FORMAT_NUMBER(row_count, 0), ' rows') AS data_volume,
  metric_order,
  metric,
  ROUND(gx_sec, 2)                               AS gx_sec,
  ROUND(dq_sec, 2)                               AS dq_sec,
  ROUND(dq_sec / gx_sec, 2)                      AS dq_vs_gx_ratio,
  CONCAT(FORMAT_NUMBER(dq_sec / gx_sec, 2), '×') AS dq_vs_gx
FROM labeled
LATERAL VIEW STACK(3,
  1, 'Inside the rules',  gx_inside,            dq_inside,
  2, 'Outside the rules', gx_total - gx_inside, dq_total - dq_inside,
  3, 'Total time',        gx_total,             dq_total
) s AS metric_order, metric, gx_sec, dq_sec;

In [0]:
%sql
CREATE OR REPLACE VIEW analytics_dq_dev.benchmark.report_rule_by_rule
COMMENT 'Benchmark report: per-rule duration, DQ Framework vs Great Expectations, per run'
AS
WITH runs AS (
  SELECT run_id, ROW_NUMBER() OVER (ORDER BY MIN(run_timestamp)) AS exec_no, MIN(run_timestamp) AS run_ts,
         MAX(row_count) AS row_count, MAX(dataset) AS dataset
  FROM analytics_dq_dev.benchmark.comparison_results
  GROUP BY run_id
),
batched AS (
  SELECT run_id, (COUNT(*) > 1 AND COUNT(DISTINCT execution_time_sec) = 1) AS dq_rules_batched
  FROM analytics_dq_dev.benchmark.comparison_results
  WHERE framework = 'dq_framework'
  GROUP BY run_id
),
pivoted AS (
  SELECT
    run_id,
    rule_name,
    MAX(CASE WHEN framework = 'great_expectations' THEN execution_time_sec END) AS gx_sec,
    MAX(CASE WHEN framework = 'dq_framework'       THEN execution_time_sec END) AS dq_sec
  FROM analytics_dq_dev.benchmark.comparison_results
  GROUP BY run_id, rule_name
)
SELECT
  r.exec_no,
  CONCAT('Exec ', r.exec_no)                         AS exec_label,
  p.run_id,
  r.run_ts,
  r.dataset,
  r.row_count,
  CONCAT(FORMAT_NUMBER(r.row_count, 0), ' rows') AS data_volume,
  p.rule_name                                        AS metric,
  ROUND(p.gx_sec, 2)                                 AS gx_sec,
  ROUND(p.dq_sec, 2)                                 AS dq_sec,
  ROUND(p.dq_sec / p.gx_sec, 2)                      AS dq_vs_gx_ratio,
  CONCAT(FORMAT_NUMBER(p.dq_sec / p.gx_sec, 2), '×') AS dq_vs_gx,
  b.dq_rules_batched                                 -- true: dq_sec = one shared scan for ALL rules
FROM pivoted p
JOIN runs r USING (run_id)
LEFT JOIN batched b USING (run_id);

In [0]:
%sql
-- ΠΡΟΣΟΧΗ: σβήνει όλο το ιστορικό των benchmark runs (η αρίθμηση Exec ξεκινά ξανά από 1).
-- TRUNCATE TABLE analytics_dq_dev.benchmark.comparison_results;